In [1]:
import random
import zipfile
from collections import defaultdict
from pathlib import Path

# Definicija osnovnih staza
DATA_ROOT = Path("/home/fipulab/anastazija/Dataset")
ZIP_FILE_PATH = DATA_ROOT / "Dikablis.zip"
OUTPUT_ROOT = Path("/home/fipulab/anastazija/TEyeD_preprocessed")

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

try:
    with zipfile.ZipFile(ZIP_FILE_PATH, 'r') as zf:
        all_files = zf.namelist()

        video_files = [f for f in all_files if "VIDEOS/" in f and not f.endswith('/')]
        annotation_files = [f for f in all_files if "ANNOTATIONS/" in f and not f.endswith('/')]

        print(f"Pronađeno video zapisa u ZIP-u: {len(video_files)}")
        print(f"Pronađeno anotacija u ZIP-u: {len(annotation_files)}")
        print(f"Izlazna mapa: {OUTPUT_ROOT}")

except FileNotFoundError:
    print(f"Greška: Datoteka {ZIP_FILE_PATH} nije pronađena.")

Pronađeno video zapisa u ZIP-u: 423
Pronađeno anotacija u ZIP-u: 9729
Izlazna mapa: /home/fipulab/anastazija/TEyeD_preprocessed


In [2]:
import json

# 1. Dohvaćanje svih mp4 videa iz ZIP-a
with zipfile.ZipFile(ZIP_FILE_PATH, 'r') as zf:
    all_files = zf.namelist()

    video_files = sorted([
        f.split("/")[-1] 
        for f in all_files 
        if "VIDEOS/" in f and f.endswith(".mp4")
    ])

print(f"Pronađeno ukupno {len(video_files)} video zapisa.")

# 2. Grupiranje videa po ID-u ispitanika
subject_to_videos = defaultdict(list)
video_to_subject = {}

for video in video_files:
    parts = video.split('_')
    if len(parts) >= 2:
        subject_id = f"{parts[0]}_{parts[1]}"  # npr. 'DikablisSA_10'
    else:
        subject_id = video  # Fallback ako ime odstupa od očekivanog formata

    subject_to_videos[subject_id].append(video)
    video_to_subject[video] = subject_id

subjects = list(subject_to_videos.keys())
print(f"Identificirano {len(subjects)} različitih ispitanika: {subjects}")

# Spremanje mapa video -> subject za kasniju upotrebu u evaluaciji (Wilcoxon test)
with open(DATA_ROOT / "subject_map.json", "w") as f:
    json.dump(video_to_subject, f, indent=4)

# 3. Miješanje ISPITANIKA (a ne pojedinačnih videa)
random.seed(42)
random.shuffle(subjects)

# 4. Podjela ispitanika na Train (70%), Val (20%), Test (10%)
n_subj = len(subjects)
train_end = int(0.7 * n_subj)
val_end = int(0.9 * n_subj)

train_subjects = subjects[:train_end]
val_subjects = subjects[train_end:val_end]
test_subjects = subjects[val_end:]

# 5. Skupljanje svih videa pripadajućih ispitanika
train_videos = [v for s in train_subjects for v in subject_to_videos[s]]
val_videos = [v for s in val_subjects for v in subject_to_videos[s]]
test_videos = [v for s in test_subjects for v in subject_to_videos[s]]

# Spremanje u txt datoteke
train_file = DATA_ROOT / "train.txt"
val_file = DATA_ROOT / "val.txt"
test_file = DATA_ROOT / "test.txt"

train_file.write_text("\n".join(train_videos))
val_file.write_text("\n".join(val_videos))
test_file.write_text("\n".join(test_videos))

print(f"Generiran train.txt: {len(train_subjects)} ispitanika ({len(train_videos)} videa)")
print(f"Generiran val.txt:   {len(val_subjects)} ispitanika ({len(val_videos)} videa)")
print(f"Generiran test.txt:  {len(test_subjects)} ispitanika ({len(test_videos)} videa)")

Pronađeno ukupno 423 video zapisa.
Identificirano 109 različitih ispitanika: ['DikablisSA_10', 'DikablisSA_11', 'DikablisSA_12', 'DikablisSA_13', 'DikablisSA_14', 'DikablisSA_15', 'DikablisSA_16', 'DikablisSA_17', 'DikablisSA_18', 'DikablisSA_19', 'DikablisSA_1', 'DikablisSA_20', 'DikablisSA_21', 'DikablisSA_22', 'DikablisSA_23', 'DikablisSA_24', 'DikablisSA_25', 'DikablisSA_26', 'DikablisSA_27', 'DikablisSA_28', 'DikablisSA_29', 'DikablisSA_2', 'DikablisSA_30', 'DikablisSA_31', 'DikablisSA_32', 'DikablisSA_33', 'DikablisSA_34', 'DikablisSA_35', 'DikablisSA_36', 'DikablisSA_37', 'DikablisSA_3', 'DikablisSA_4', 'DikablisSA_5', 'DikablisSA_6', 'DikablisSA_7', 'DikablisSA_8', 'DikablisSA_9', 'DikablisSS_10', 'DikablisSS_11', 'DikablisSS_12', 'DikablisSS_13', 'DikablisSS_14', 'DikablisSS_15', 'DikablisSS_16', 'DikablisSS_17', 'DikablisSS_18', 'DikablisSS_19', 'DikablisSS_1', 'DikablisSS_20', 'DikablisSS_21', 'DikablisSS_22', 'DikablisSS_23', 'DikablisSS_24', 'DikablisSS_25', 'DikablisSS_26

### Očuvanje i normalizacija izvornih 3D gaze anotacija

Predobrada čuva izvorne komponente `X`, `Y` i `Z` iz `gaze_vec.txt` (bez rekonstrukcije `Z` iz `X,Y`). Nakon odbacivanja nevaljanih `(-1,-1,-1)` redaka i redaka s nul-vektorom, svaki preostali 3D vektor normalizira se na jediničnu duljinu. `labels.csv` sadrži normalizirane `x,y,z`, `subject_id`, `video_id` i `mask_valid_<part>` stupce. Nedostupne segmentacijske maske imaju `mask_valid_<part>=0` i moraju se ignorirati u segmentacijskom gubitku.

In [4]:
import pandas as pd
import numpy as np

# Staze za otpakirani dataset i skriptu
EXTRACTED_DATA_ROOT = DATA_ROOT / "Dikablis" / "Dikablis"
SPLITS_DIR = DATA_ROOT
SCRIPT_PATH = Path("/home/fipulab/anastazija/teyed-utils/preprocess_teyed.py")

# Originalna skripta odbacuje gaze uzorke ako nedostaje bilo koja od tri maske.
# Ovdje radimo privremenu kopiju koja čuva gaze podatke i bilježi valjanost maski.
import tempfile
import subprocess
from PIL import Image

original_script = SCRIPT_PATH.read_text(encoding="utf-8")

# Ispis napretka prije obrade svakog videa (u izvornoj funkciji process_split).
original_script = original_script.replace(
    '    for idx, name in enumerate(video_list, start=1):\n        result = process_video(',
    '    for idx, name in enumerate(video_list, start=1):\n'
    '        print(f"  [{idx}/{len(video_list)}] ZAPOČINJEM: {name}", flush=True)\n'
    '        result = process_video(',
    1,
)
# Preserve all three components of the original gaze vector (X,Y,Z).
original_script = original_script.replace(
    'usecols=[0, 1, 2],\n            names=["frame", "x", "y"],',
    'usecols=[0, 1, 2, 3],\n            names=["frame", "x", "y", "z"],',
)
original_script = original_script.replace(
    'df = df[(df["x"] != -1) & (df["y"] != -1)].copy()',
    'df = df[df[["x", "y", "z"]].notna().all(axis=1) & ~df[["x", "y", "z"]].eq(-1).all(axis=1)].copy()',
)
# Normalize the retained original XYZ vectors; discard zero-length rows.
normalization_code = """\ndf[["x", "y", "z"]] = df[["x", "y", "z"]].apply(pd.to_numeric, errors="coerce")\ndf = df[df[["x", "y", "z"]].notna().all(axis=1)].copy()\n_gaze = df[["x", "y", "z"]].to_numpy(dtype=float)\n_gaze_norm = np.linalg.norm(_gaze, axis=1)\ndf = df.loc[_gaze_norm > 0].copy()\n_gaze_norm = _gaze_norm[_gaze_norm > 0]\ndf["norm_before"] = _gaze_norm\ndf[["x", "y", "z"]] = df[["x", "y", "z"]].to_numpy(dtype=float) / _gaze_norm[:, None]\n"""
normalization_code = "\n".join(
    ("        " + line) if line else line
    for line in normalization_code.splitlines()
) + "\n"
original_script = original_script.replace(
    'df = df[df[["x", "y", "z"]].notna().all(axis=1) & ~df[["x", "y", "z"]].eq(-1).all(axis=1)].copy()',
    'df = df[df[["x", "y", "z"]].notna().all(axis=1) & ~df[["x", "y", "z"]].eq(-1).all(axis=1)].copy()\n' + normalization_code,
    1,
)
original_script = original_script.replace(
    "import pandas as pd", "import pandas as pd\nimport numpy as np\nfrom PIL import Image"
)
# Očisti stare slike prije ponovne predobrade kako se ne bi računali zastarjeli uzorci.
original_script = original_script.replace(
    "    out_dir.mkdir(parents=True, exist_ok=True)\n    w, h = size",
    "    out_dir.mkdir(parents=True, exist_ok=True)\n    for old_file in out_dir.glob('*.jpg'):\n        old_file.unlink()\n    w, h = size", 1
)
block_start = original_script.index("        if segmentation:\n            seg_dirs = {}")
block_end = original_script.index("\n        if df.empty:", block_start)
replacement = '''        mask_available = {part: set() for part in seg_parts}
        if segmentation:
            for seg_part in seg_parts:
                seg_path = annotations_dir / f"{name}{seg_part}_seg_{seg_dimension}.mp4"
                seg_dir = out_root / f"seg_{seg_part}_{seg_dimension}" / vpath.stem
                seg_dir.mkdir(parents=True, exist_ok=True)
                for old_mask in seg_dir.glob("*.png"):
                    old_mask.unlink()
                if not seg_path.exists():
                    print(f"    mask video missing: {seg_path.name}; keeping gaze labels")
                    continue
                extract_segmentation_ffmpeg(
                    seg_path, seg_dir, size=(frame_width, frame_height),
                    stride=frame_stride, binarize=seg_binarize, threshold=seg_threshold,
                )
                mask_available[seg_part] = list_frame_indices(seg_dir, ".png")

            # Keep all gaze labels, even when a segmentation mask is missing.
            # A zero placeholder is paired with mask_valid_<part>=0, so it must
            # not be treated as a real segmentation annotation during training.
            for seg_part in seg_parts:
                seg_dir = out_root / f"seg_{seg_part}_{seg_dimension}" / vpath.stem
                seg_dir.mkdir(parents=True, exist_ok=True)
                valid_indices = mask_available[seg_part]
                df[f"mask_valid_{seg_part}"] = df["img_index"].isin(valid_indices).astype(int)
                for img_index in df["img_index"].astype(int):
                    mask_path = seg_dir / f"{img_index:06d}.png"
                    if not mask_path.exists():
                        Image.new("L", (frame_width, frame_height), 0).save(mask_path)
'''
original_script = original_script[:block_start] + replacement + original_script[block_end:]
old_rows = '''        rows = []
        for frame, img_index, x, y in df[["frame", "img_index", "x", "y"]].itertuples(
            index=False, name=None
        ):
            rel = f"{vpath.stem}/{int(img_index):06d}.jpg"
            if not (out_dir / f"{int(img_index):06d}.jpg").exists():
                continue
            rows.append((rel, float(x), float(y)))
'''
new_rows = '''        rows = []
        row_columns = ["frame", "img_index", "x", "y", "z", "norm_before"]
        if segmentation:
            row_columns += [f"mask_valid_{part}" for part in seg_parts]
        for values in df[row_columns].itertuples(index=False, name=None):
            frame, img_index, x, y, z, norm_before = values[:6]
            validity = values[6:]
            rel = f"{vpath.stem}/{int(img_index):06d}.jpg"
            if not (out_dir / f"{int(img_index):06d}.jpg").exists():
                continue
            rows.append((rel, float(x), float(y), float(z), float(norm_before), *[int(v) for v in validity]))
'''
if old_rows not in original_script:
    raise RuntimeError("Could not find rows block to patch")
original_script = original_script.replace(old_rows, new_rows, 1)
old_writer = '''            pd.DataFrame(all_rows, columns=["filename", "x", "y"]).to_csv(
                out_csv, index=False
            )'''
new_writer = '''            output_columns = ["filename", "x", "y", "z", "norm_before"]
            if args.segmentation:
                output_columns += [f"mask_valid_{part}" for part in seg_parts]
            pd.DataFrame(all_rows, columns=output_columns).to_csv(out_csv, index=False)'''
if old_writer not in original_script:
    raise RuntimeError("Could not find CSV writer block to patch")
original_script = original_script.replace(old_writer, new_writer, 1)
patched_script = Path(tempfile.gettempdir()) / "preprocess_teyed_keep_all.py"
patched_script.write_text(original_script, encoding="utf-8")
print(f"Pokrećem privremenu skriptu: {patched_script}")
print("Čuvaju se svi gaze uzorci; nedostajuće maske imaju mask_valid_<dio>=0.")
cmd = [
    "python3", "-u", str(patched_script),
    "--data_root", str(EXTRACTED_DATA_ROOT),
    "--output_root", str(OUTPUT_ROOT),
    "--splits_dir", str(SPLITS_DIR),
    "--frame_width", "128", "--frame_height", "128",
    "--frame_stride", "10", "--jpeg_q", "4",
    "--train_sample_frac", "1.0", "--val_sample_frac", "1.0", "--test_sample_frac", "1.0",
    "--segmentation", "--seg_parts", "pupil,iris,lid",
    "--seg_dimension", "2D", "--seg_binarize",
]
print("Pokretanje predobrade; ispis napretka slijedi uživo...", flush=True)
process = subprocess.Popen(
    cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1,
)
for line in process.stdout:
    print(line, end="", flush=True)
return_code = process.wait()
if return_code != 0:
    raise subprocess.CalledProcessError(return_code, cmd)

# Dodavanje identifikatora ispitanika i videa, validacija stupaca te metapodaci.
processed_dir = OUTPUT_ROOT / "128x128_stride10_q4_seg_pupil-iris-lid_2D"
split_summary = {}
required_columns = [
    "filename", "x", "y", "z", "subject_id", "video_id",
    "mask_valid_pupil", "mask_valid_iris", "mask_valid_lid",
]

def extract_video_stem(filename):
    parts = Path(str(filename)).parts
    return parts[-2] if len(parts) >= 2 else Path(str(filename)).parent.name

def extract_subject(video_stem):
    # Primjer: DikablisT_5_1 -> DikablisT_5.
    parts = str(video_stem).split("_")
    return f"{parts[0]}_{parts[1]}" if len(parts) >= 2 else "unknown"

for split in ["train", "val", "test"]:
    csv_path = processed_dir / split / "labels.csv"
    if not csv_path.exists():
        raise FileNotFoundError(f"Nedostaje labels.csv za split {split}: {csv_path}")
    df = pd.read_csv(csv_path)
    if "filename" not in df.columns:
        raise ValueError(f"{csv_path} nema stupac filename; pronađeni: {list(df.columns)}")

    df["video_id"] = df["filename"].apply(extract_video_stem)
    df["subject_id"] = df["video_id"].apply(extract_subject)

    if "norm_before" not in df.columns:
        raise ValueError(f"{csv_path} nema norm_before; provjeri normalizaciju u skripti.")
    before = pd.to_numeric(df["norm_before"], errors="coerce").to_numpy(dtype=float)
    vectors = df[["x", "y", "z"]].to_numpy(dtype=float)
    after = np.linalg.norm(vectors, axis=1)
    if len(after) and not np.allclose(after, 1.0, atol=1e-6):
        raise ValueError(f"{split}: pronađeni XYZ vektori koji nisu jedinične duljine.")
    if len(vectors) and not np.isfinite(vectors).all():
        raise ValueError(f"{split}: XYZ sadrži NaN ili beskonačne vrijednosti.")

    df = df.drop(columns=["norm_before"])
    missing = set(required_columns) - set(df.columns)
    if missing:
        raise ValueError(f"{csv_path} nema obavezne stupce: {sorted(missing)}")
    df.to_csv(csv_path, index=False)

    split_summary[split] = {
        "subject_ids": sorted(df["subject_id"].dropna().unique().tolist()),
        "num_samples": int(len(df)),
        "num_videos": int(df["video_id"].nunique()),
        "mask_valid_counts": {
            part: int(df[f"mask_valid_{part}"].fillna(0).astype(int).sum())
            for part in ["pupil", "iris", "lid"]
        },
        "norm_before_min": float(np.min(before)) if len(before) else None,
        "norm_before_max": float(np.max(before)) if len(before) else None,
        "norm_after_min": float(np.min(after)) if len(after) else None,
        "norm_after_max": float(np.max(after)) if len(after) else None,
    }

metadata = {
    "preprocessing_version": "v2_normalized",
    "seed": 42,
    "split_subject_ids": {s: split_summary[s]["subject_ids"] for s in ["train", "val", "test"]},
    "num_samples_per_split": {s: split_summary[s]["num_samples"] for s in ["train", "val", "test"]},
    "num_videos_per_split": {s: split_summary[s]["num_videos"] for s in ["train", "val", "test"]},
}
with open(OUTPUT_ROOT / "metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=4, ensure_ascii=False)

print("\nSAŽETAK PREDOBRADE PO SPLITU")
for split in ["train", "val", "test"]:
    item = split_summary[split]
    print(f"{split}: uzoraka={item['num_samples']}, ispitanika={len(item['subject_ids'])}, videa={item['num_videos']}")
    for part in ["pupil", "iris", "lid"]:
        print(f"  maska {part}: {item['mask_valid_counts'][part]}/{item['num_samples']}")
    print(f"  norma prije: min={item['norm_before_min']}, max={item['norm_before_max']}; "
          f"poslije: min={item['norm_after_min']}, max={item['norm_after_max']}")
print(f"Metapodaci spremljeni: {OUTPUT_ROOT / 'metadata.json'}")
print(f"Mapa video->subject (postojeća lokacija): {DATA_ROOT / 'subject_map.json'}")


Pokrećem privremenu skriptu: /tmp/preprocess_teyed_keep_all.py
Čuvaju se svi gaze uzorci; nedostajuće maske imaju mask_valid_<dio>=0.
Pokretanje predobrade; ispis napretka slijedi uživo...
Saving preprocessed data to: /home/fipulab/anastazija/TEyeD_preprocessed/128x128_stride10_q4_seg_pupil-iris-lid_2D
Processing 265 videos for split train...
  [1/265] ZAPOČINJEM: DikablisSS_32_1.mp4
  [1/265] DikablisSS_32_1.mp4: frames_saved=221, labels_used=220
  [2/265] ZAPOČINJEM: DikablisSA_19_1.mp4
  [2/265] DikablisSA_19_1.mp4: frames_saved=4681, labels_used=4498
  [3/265] ZAPOČINJEM: DikablisSA_11_1.mp4
  [3/265] DikablisSA_11_1.mp4: frames_saved=4959, labels_used=4652
  [4/265] ZAPOČINJEM: DikablisSS_22_1.mp4
  [4/265] DikablisSS_22_1.mp4: frames_saved=180, labels_used=179
  [5/265] ZAPOČINJEM: DikablisT_5_1.mp4
  [5/265] DikablisT_5_1.mp4: frames_saved=5219, labels_used=5176
  [6/265] ZAPOČINJEM: DikablisT_5_10.mp4
  [6/265] DikablisT_5_10.mp4: frames_saved=75, labels_used=75
  [7/265] ZAPOČ

  [72/265] ZAPOČINJEM: DikablisT_18_9.mp4
  [72/265] DikablisT_18_9.mp4: frames_saved=75, labels_used=75
  [73/265] ZAPOČINJEM: DikablisT_16_1.mp4
  [73/265] DikablisT_16_1.mp4: frames_saved=4637, labels_used=4521
  [74/265] ZAPOČINJEM: DikablisT_16_2.mp4
  [74/265] DikablisT_16_2.mp4: frames_saved=5091, labels_used=4923
  [75/265] ZAPOČINJEM: DikablisT_16_3.mp4
  [75/265] DikablisT_16_3.mp4: frames_saved=368, labels_used=354
  [76/265] ZAPOČINJEM: DikablisT_16_4.mp4
  [76/265] DikablisT_16_4.mp4: frames_saved=538, labels_used=503
  [77/265] ZAPOČINJEM: DikablisT_12_1.mp4
  [77/265] DikablisT_12_1.mp4: frames_saved=5343, labels_used=5309
  [78/265] ZAPOČINJEM: DikablisT_12_10.mp4
  [78/265] DikablisT_12_10.mp4: frames_saved=75, labels_used=75
  [79/265] ZAPOČINJEM: DikablisT_12_11.mp4
  [79/265] DikablisT_12_11.mp4: frames_saved=75, labels_used=75
  [80/265] ZAPOČINJEM: DikablisT_12_12.mp4
  [80/265] DikablisT_12_12.mp4: frames_saved=75, labels_used=75
  [81/265] ZAPOČINJEM: DikablisT_

    Last ffmpeg message: /home/fipulab/anastazija/Dataset/Dikablis/Dikablis/ANNOTATIONS/DikablisT_9_13.mp4lid_seg_2D.mp4: Invalid data found when processing input
  [141/265] DikablisT_9_13.mp4: frames_saved=75, labels_used=75
  [142/265] ZAPOČINJEM: DikablisT_9_2.mp4
  [142/265] DikablisT_9_2.mp4: frames_saved=6062, labels_used=5984
  [143/265] ZAPOČINJEM: DikablisT_9_3.mp4
  [143/265] DikablisT_9_3.mp4: frames_saved=204, labels_used=201
  [144/265] ZAPOČINJEM: DikablisT_9_4.mp4
    ffmpeg reported errors for DikablisT_9_4.mp4pupil_seg_2D.mp4 (return code 1). Proceeding with any frames decoded before the failure.
    Last ffmpeg message: /home/fipulab/anastazija/Dataset/Dikablis/Dikablis/ANNOTATIONS/DikablisT_9_4.mp4pupil_seg_2D.mp4: Invalid data found when processing input
  [144/265] DikablisT_9_4.mp4: frames_saved=75, labels_used=73
  [145/265] ZAPOČINJEM: DikablisT_9_5.mp4
    ffmpeg reported errors for DikablisT_9_5.mp4pupil_seg_2D.mp4 (return code 1). Proceeding with any frames 

  [196/265] ZAPOČINJEM: DikablisT_21_11.mp4
  [196/265] DikablisT_21_11.mp4: frames_saved=75, labels_used=74
  [197/265] ZAPOČINJEM: DikablisT_21_12.mp4
  [197/265] DikablisT_21_12.mp4: frames_saved=75, labels_used=72
  [198/265] ZAPOČINJEM: DikablisT_21_2.mp4
  [198/265] DikablisT_21_2.mp4: frames_saved=6555, labels_used=6384
  [199/265] ZAPOČINJEM: DikablisT_21_3.mp4
  [199/265] DikablisT_21_3.mp4: frames_saved=75, labels_used=75
  [200/265] ZAPOČINJEM: DikablisT_21_4.mp4
  [200/265] DikablisT_21_4.mp4: frames_saved=75, labels_used=74
  [201/265] ZAPOČINJEM: DikablisT_21_5.mp4
  [201/265] DikablisT_21_5.mp4: frames_saved=75, labels_used=74
  [202/265] ZAPOČINJEM: DikablisT_21_6.mp4
  [202/265] DikablisT_21_6.mp4: frames_saved=75, labels_used=71
  [203/265] ZAPOČINJEM: DikablisT_21_7.mp4
  [203/265] DikablisT_21_7.mp4: frames_saved=75, labels_used=74
  [204/265] ZAPOČINJEM: DikablisT_21_8.mp4
  [204/265] DikablisT_21_8.mp4: frames_saved=75, labels_used=73
  [205/265] ZAPOČINJEM: Dikab


Processing 108 videos for split val...
  [1/108] ZAPOČINJEM: DikablisSA_10_1.mp4
  [1/108] DikablisSA_10_1.mp4: frames_saved=5022, labels_used=4963
  [2/108] ZAPOČINJEM: DikablisSA_10_2.mp4
  [2/108] DikablisSA_10_2.mp4: frames_saved=994, labels_used=983
  [3/108] ZAPOČINJEM: DikablisT_6_1.mp4
  [3/108] DikablisT_6_1.mp4: frames_saved=5253, labels_used=5236
  [4/108] ZAPOČINJEM: DikablisT_6_10.mp4
  [4/108] DikablisT_6_10.mp4: frames_saved=75, labels_used=75
  [5/108] ZAPOČINJEM: DikablisT_6_11.mp4
  [5/108] DikablisT_6_11.mp4: frames_saved=75, labels_used=75
  [6/108] ZAPOČINJEM: DikablisT_6_12.mp4
  [6/108] DikablisT_6_12.mp4: frames_saved=75, labels_used=74
  [7/108] ZAPOČINJEM: DikablisT_6_13.mp4
  [7/108] DikablisT_6_13.mp4: frames_saved=75, labels_used=75
  [8/108] ZAPOČINJEM: DikablisT_6_2.mp4
  [8/108] DikablisT_6_2.mp4: frames_saved=5928, labels_used=5894
  [9/108] ZAPOČINJEM: DikablisT_6_3.mp4
  [9/108] DikablisT_6_3.mp4: frames_saved=13, labels_used=13
  [10/108] ZAPOČINJEM

  [59/108] DikablisT_24_2.mp4: frames_saved=5215, labels_used=5138
  [60/108] ZAPOČINJEM: DikablisT_24_3.mp4
  [60/108] DikablisT_24_3.mp4: frames_saved=136, labels_used=136
  [61/108] ZAPOČINJEM: DikablisT_24_4.mp4
  [61/108] DikablisT_24_4.mp4: frames_saved=929, labels_used=907
  [62/108] ZAPOČINJEM: DikablisT_24_5.mp4
  [62/108] DikablisT_24_5.mp4: frames_saved=75, labels_used=73
  [63/108] ZAPOČINJEM: DikablisT_24_7.mp4
  [63/108] DikablisT_24_7.mp4: frames_saved=75, labels_used=75
  [64/108] ZAPOČINJEM: DikablisT_24_8.mp4
  [64/108] DikablisT_24_8.mp4: frames_saved=75, labels_used=74
  [65/108] ZAPOČINJEM: DikablisT_24_9.mp4
  [65/108] DikablisT_24_9.mp4: frames_saved=75, labels_used=75
  [66/108] ZAPOČINJEM: DikablisT_11_1.mp4
  [66/108] DikablisT_11_1.mp4: frames_saved=5219, labels_used=5133
  [67/108] ZAPOČINJEM: DikablisT_11_10.mp4
  [67/108] DikablisT_11_10.mp4: frames_saved=75, labels_used=73
  [68/108] ZAPOČINJEM: DikablisT_11_11.mp4
  [68/108] DikablisT_11_11.mp4: frames_s

  [20/50] ZAPOČINJEM: DikablisT_7_3.mp4
  [20/50] DikablisT_7_3.mp4: frames_saved=75, labels_used=75
  [21/50] ZAPOČINJEM: DikablisT_7_4.mp4
  [21/50] DikablisT_7_4.mp4: frames_saved=75, labels_used=75
  [22/50] ZAPOČINJEM: DikablisT_7_5.mp4
  [22/50] DikablisT_7_5.mp4: frames_saved=75, labels_used=75
  [23/50] ZAPOČINJEM: DikablisT_7_6.mp4
  [23/50] DikablisT_7_6.mp4: frames_saved=75, labels_used=74
  [24/50] ZAPOČINJEM: DikablisT_7_7.mp4
  [24/50] DikablisT_7_7.mp4: frames_saved=75, labels_used=75
  [25/50] ZAPOČINJEM: DikablisT_7_8.mp4
  [25/50] DikablisT_7_8.mp4: frames_saved=75, labels_used=74
  [26/50] ZAPOČINJEM: DikablisT_7_9.mp4
  [26/50] DikablisT_7_9.mp4: frames_saved=75, labels_used=73
  [27/50] ZAPOČINJEM: DikablisSA_26_1.mp4
  [27/50] DikablisSA_26_1.mp4: frames_saved=5158, labels_used=5044
  [28/50] ZAPOČINJEM: DikablisSA_26_2.mp4
  [28/50] DikablisSA_26_2.mp4: frames_saved=251, labels_used=248
  [29/50] ZAPOČINJEM: DikablisSA_36_1.mp4
  [29/50] DikablisSA_36_1.mp4: fram